# 01 - Train Forecast Model

Goal: predict `production_next_quarter` from `gold.crop_forecast_features`.

1. Split the data **by time**: train <= 2022, validation 2023-2024, test 2025
2. Compute a simple **baseline** (next quarter = same quarter last year). Every real model must beat it.
3. Train three models: linear regression, random forest, LightGBM
4. Register the model with the lowest validation MAE in the **Unity Catalog Model Registry**
   and give that version the alias `champion` (MLflow is used only for this)

In [0]:
%pip install lightgbm --quiet

In [0]:
dbutils.library.restartPython()

## 1. Configuration

In [0]:
import mlflow
import numpy as np
from lightgbm import LGBMRegressor
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from configs.ml_config import (
    ALL_FEATURE_COLUMNS,
    BOOLEAN_FEATURES,
    CATEGORICAL_FEATURES,
    FEATURE_TABLE,
    LABEL_COLUMN,
    MODEL_ALIAS,
    NUMERIC_FEATURES,
    REGISTERED_MODEL_NAME,
    TEST_YEAR,
    TRAIN_END_YEAR,
    VALIDATION_YEARS,
)

RANDOM_SEED = 42

## 2. Load labeled rows and split by year

In [0]:
labeled_df = spark.table(FEATURE_TABLE).where(f"{LABEL_COLUMN} IS NOT NULL").toPandas()

# Plain float columns keep the model's input format simple (also for Model Serving, where JSON has no int/NULL mix)
for column in NUMERIC_FEATURES + BOOLEAN_FEATURES:
    labeled_df[column] = labeled_df[column].astype("float")

train_df = labeled_df[labeled_df["year"] <= TRAIN_END_YEAR]
validation_df = labeled_df[labeled_df["year"].isin(VALIDATION_YEARS)]
test_df = labeled_df[labeled_df["year"] == TEST_YEAR]
print(f"train: {len(train_df):,} | validation: {len(validation_df):,} | test: {len(test_df):,}")

if train_df.empty or validation_df.empty:
    raise ValueError("Train or validation split is empty. Check the year settings in configs/ml_config.py")

X_train, y_train = train_df[ALL_FEATURE_COLUMNS], train_df[LABEL_COLUMN]
X_validation, y_validation = validation_df[ALL_FEATURE_COLUMNS], validation_df[LABEL_COLUMN]
X_test, y_test = test_df[ALL_FEATURE_COLUMNS], test_df[LABEL_COLUMN]

## 3. Helper functions

In [0]:
def regression_metrics(actual, predicted) -> dict:
    """MAE, RMSE, MAPE and R2. MAPE ignores rows where the actual value is 0."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    non_zero = actual != 0
    mape = (
        float(np.mean(np.abs((actual[non_zero] - predicted[non_zero]) / actual[non_zero])))
        if non_zero.any()
        else None
    )
    return {
        "mae": float(mean_absolute_error(actual, predicted)),
        "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
        "mape": mape,
        "r2": float(r2_score(actual, predicted)),
    }


def build_preprocessor() -> ColumnTransformer:
    """Fill missing values, then one-hot encode the text columns."""
    numeric_steps = Pipeline([("fill_missing", SimpleImputer(strategy="median"))])
    categorical_steps = Pipeline([
        ("fill_missing", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_steps, NUMERIC_FEATURES + BOOLEAN_FEATURES),
        ("categorical", categorical_steps, CATEGORICAL_FEATURES),
    ])


candidate_models = {
    "linear_regression": Pipeline([
        ("preprocess", build_preprocessor()),
        ("model", LinearRegression()),
    ]),
    "random_forest": Pipeline([
        ("preprocess", build_preprocessor()),
        ("model", RandomForestRegressor(
            n_estimators=100, max_depth=12, min_samples_leaf=5,
            n_jobs=-1, random_state=RANDOM_SEED,
        )),
    ]),
    "lightgbm": Pipeline([
        ("preprocess", build_preprocessor()),
        ("model", LGBMRegressor(
            n_estimators=200, learning_rate=0.05, num_leaves=31,
            random_state=RANDOM_SEED, verbosity=-1,
        )),
    ]),
}

## 4. Baseline: same quarter last year

For a row at quarter t the target is quarter t+1, and its "same quarter last year" is `production_lag_3`.

In [0]:
baseline_metrics = regression_metrics(y_validation, X_validation["production_lag_3"].fillna(0.0))
print(f"baseline_same_quarter_last_year   validation MAE = {baseline_metrics['mae']:,.3f}")

## 5. Train and compare the models

In [0]:
trained_models = {}
validation_mae = {}

for model_name, model in candidate_models.items():
    model.fit(X_train, y_train)
    validation_scores = regression_metrics(y_validation, model.predict(X_validation))
    trained_models[model_name] = model
    validation_mae[model_name] = validation_scores["mae"]

    print(f"{model_name:34}validation: " + ", ".join(
        f"{k}={v:,.3f}" for k, v in validation_scores.items() if v is not None
    ))
    if not test_df.empty:
        test_scores = regression_metrics(y_test, model.predict(X_test))
        print(f"{'':34}test      : mae={test_scores['mae']:,.3f}, r2={test_scores['r2']:,.3f}")

## 6. Register the winner in Unity Catalog

Each run adds a new model **version**. The alias `champion` is moved to the newest version.

In [0]:
winner_name = min(validation_mae, key=validation_mae.get)
print(f"Winner (lowest validation MAE): {winner_name}")

if validation_mae[winner_name] >= baseline_metrics["mae"]:
    print("WARNING: no model beats the baseline. Check the features before using this model.")

mlflow.set_registry_uri("databricks-uc")

winner_model = trained_models[winner_name]
signature = infer_signature(X_train, winner_model.predict(X_train))   # required by Unity Catalog

with mlflow.start_run(run_name=f"register_{winner_name}"):
    mlflow.log_param("algorithm", winner_name)
    mlflow.log_metric("validation_mae", validation_mae[winner_name])
    model_info = mlflow.sklearn.log_model(
        winner_model,
        "model",
        signature=signature,
        registered_model_name=REGISTERED_MODEL_NAME,
        # MLflow 3 defaults to the "skops" format, which rejects random forests. cloudpickle works everywhere.
        serialization_format="cloudpickle",
    )

MlflowClient().set_registered_model_alias(
    REGISTERED_MODEL_NAME, MODEL_ALIAS, model_info.registered_model_version
)
print(f"Registered {REGISTERED_MODEL_NAME} version {model_info.registered_model_version} (alias: {MODEL_ALIAS})")
print("Next: 02_evaluate_model")